# FLN Specific Level Populator
Scrape images for specific levels only. Change `LEVELS_TO_SCRAPE` below to target the levels you need.

In [ ]:
# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["duckduckgo_search", "requests", "Pillow", "imagehash"]:
    try:
        __import__(pkg.replace("-", "_"))
        print(f"  OK {pkg}")
    except ImportError:
        print(f"  Installing {pkg}...")
        install(pkg)
print("Done.")

In [ ]:
# ============================================================
# INTERACTIVE CONFIGURATION (answer in terminal)
# ============================================================
from pathlib import Path

BASE = Path(input("BASE path [default: /content/fln_output]: ").strip() or "/content/fln_output")
IMAGES_PER_SUBLEVEL = int(input("Images per sub-level [default: 12]: ").strip() or "12")

levels_in = input("Level numbers to scrape (comma-sep) [default: 23,24,25,26,28,29,30,31,32]: ").strip()
if not levels_in:
    levels_in = "23,24,25,26,28,29,30,31,32"
LEVELS_TO_SCRAPE = [int(x.strip()) for x in levels_in.split(",") if x.strip()]

review_in = input("Review assessment levels with source ranges (e.g. 23:12-22) or empty if none: ").strip()
REVIEW_ASSESSMENT_LEVELS = {}
if review_in:
    for part in review_in.split(","):
        part = part.strip()
        if ":" in part and "-" in part:
            lvl, rng = part.split(":")
            s, e = rng.split("-")
            REVIEW_ASSESSMENT_LEVELS[int(lvl.strip())] = (int(s.strip()), int(e.strip()))

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
print(f"\nConfig loaded: {len(LEVELS_TO_SCRAPE)} levels, {len(REVIEW_ASSESSMENT_LEVELS)} review levels")

In [ ]:
# ============================================================
# SEARCH QUERIES PER LEVEL
# ============================================================
SEARCH_QUERIES = {
 "24": {
  "name": "Place Value / Numbers 51-100",
  "queries": [
   "numbers 51-100 worksheet",
   "counting 51-100 worksheet",
   "missing numbers 51-100",
   "number chart 51 to 100",
   "write numbers 51-100",
   "number recognition 51-100",
   "count to 100 worksheet",
   "number sequencing 51-100",
   "hundred chart activities",
   "numbers to 100 worksheet"
  ]
 },
 "25": {
  "name": "Carry Addition",
  "queries": [
   "carry addition worksheet",
   "addition with regrouping worksheet",
   "2 digit addition with carry",
   "vertical addition with regrouping",
   "addition carry over worksheet",
   "adding with regrouping 2 digit",
   "carry addition sums worksheet",
   "addition regrouping practice",
   "two digit addition with carry",
   "addition with carry grade 2"
  ]
 },
 "26": {
  "name": "Borrow Subtraction",
  "queries": [
   "borrow subtraction worksheet",
   "subtraction with regrouping worksheet",
   "2 digit subtraction with borrow",
   "vertical subtraction with regrouping",
   "subtraction borrow over worksheet",
   "subtracting with regrouping 2 digit",
   "borrow subtraction sums worksheet",
   "subtraction regrouping practice",
   "two digit subtraction with borrow",
   "subtraction with borrowing grade 2"
  ]
 },
 "27": {
  "name": "Comparison Up to 100",
  "queries": [
   "compare numbers to 100 worksheet",
   "greater than less than 2 digit",
   "comparing 2 digit numbers",
   "compare numbers using symbols",
   "more less numbers to 100",
   "number comparison grade 2",
   "compare numbers whole class",
   "greater less equal numbers",
   "comparing numbers to 100",
   "number inequality worksheet"
  ]
 },
 "28": {
  "name": "Ordering Ascend Descend",
  "queries": [
   "ascending descending order worksheet",
   "ordering numbers to 100",
   "least greatest numbers worksheet",
   "arrange numbers ascending",
   "arrange numbers descending",
   "number order worksheet grade 2",
   "ordering 3 sets",
   "smallest largest number worksheet",
   "numerical order worksheet",
   "sequencing numbers to 100"
  ]
 },
 "29": {
  "name": "Tally Marks",
  "queries": [
   "tally marks worksheet",
   "tally chart worksheet",
   "count tally marks worksheet",
   "tally mark activity",
   "draw tally marks worksheet",
   "tally graph worksheet",
   "reading tally marks",
   "tally counting worksheet",
   "data handling tally marks",
   "tally marks kindergarten"
  ]
 },
 "30": {
  "name": "Time Analog Clock",
  "queries": [
   "analog clock worksheet",
   "tell time worksheet",
   "reading clock worksheet",
   "clock practice worksheet",
   "hour half hour worksheet",
   "time to the hour worksheet",
   "time to half hour worksheet",
   "clock faces worksheet",
   "telling time analog",
   "what time is it worksheet"
  ]
 },
 "31": {
  "name": "Ordinal Positions",
  "queries": [
   "ordinal numbers worksheet",
   "ordinal position worksheet",
   "1st to 10th worksheet",
   "ordinal numbers kindergarten",
   "positional words worksheet",
   "first second third worksheet",
   "order position worksheet",
   "ordinal number activity",
   "sequence position worksheet",
   "ordinal words worksheet"
  ]
 },
 "32": {
  "name": "Multiplication Repeated Add",
  "queries": [
   "multiplication repeated addition worksheet",
   "equal groups worksheet",
   "repeated addition worksheet",
   "multiplication as repeated addition",
   "equal groups multiplication",
   "intro to multiplication worksheet",
   "multiply with pictures worksheet",
   "count equal groups worksheet",
   "beginning multiplication worksheet",
   "arrays repeated addition"
  ]
 }
}


In [ ]:
# ============================================================
# IMAGE DOWNLOAD UTILITIES
# ============================================================
import os, io, time, requests, shutil
from PIL import Image

MIN_IMAGE_SIZE = 10000
MAX_IMAGE_SIZE = 5000000

def safe_filename(url, index):
    ext = os.path.splitext(url.split("?")[0])[1].lower()
    if ext not in VALID_EXTENSIONS:
        ext = ".jpg"
    return f"{index:03d}{ext}"

def download_image(url, dest_path, timeout=15):
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        r = requests.get(url, headers=headers, timeout=timeout, stream=True)
        r.raise_for_status()
        content = r.content
        if len(content) < MIN_IMAGE_SIZE or len(content) > MAX_IMAGE_SIZE:
            return False
        try:
            img = Image.open(io.BytesIO(content))
            img.verify()
            img = Image.open(io.BytesIO(content))
            if img.width < 100 or img.height < 100:
                return False
            aspect = max(img.width, img.height) / max(min(img.width, img.height), 1)
            if aspect > 4.0:
                return False
        except Exception:
            return False
        with open(dest_path, "wb") as f:
            f.write(content)
        return True
    except Exception:
        return False

def deduplicate(dir_path):
    from PIL import Image
    import imagehash
    hashes = {}
    removed = 0
    for fname in sorted(os.listdir(dir_path)):
        fpath = os.path.join(dir_path, fname)
        try:
            h = imagehash.average_hash(Image.open(fpath))
            if h in hashes:
                os.remove(fpath)
                removed += 1
            else:
                hashes[h] = fpath
        except Exception:
            pass
    return removed

def deduplicate_across_subdirs(parent_dir):
    removed_total = 0
    subdirs = sorted([d for d in parent_dir.iterdir() if d.is_dir()])
    seen_hashes = set()
    import imagehash
    for subdir in subdirs:
        for fname in sorted(os.listdir(subdir)):
            fpath = os.path.join(subdir, fname)
            try:
                h = imagehash.average_hash(Image.open(fpath))
                if h in seen_hashes:
                    os.remove(fpath)
                    removed_total += 1
                else:
                    seen_hashes.add(h)
            except Exception:
                pass
    return removed_total

In [ ]:
# ============================================================
# SCRAPER 1: DuckDuckGo
# ============================================================
def scrape_duckduckgo(search_term, dest_dir, max_images=12):
    try:
        from duckduckgo_search import DDGS
        existing = set(os.listdir(dest_dir)) if os.path.exists(dest_dir) else set()
        count = 0
        with DDGS() as ddgs:
            results = ddgs.images(search_term, region="wt-wt", safesearch="off",
                                  max_results=max_images * 2)
            for result in results:
                if count >= max_images:
                    break
                url = result.get("image", "")
                if not url:
                    continue
                fname = safe_filename(url, len(existing) + count + 1)
                dst = dest_dir / fname
                if dst.exists():
                    continue
                if download_image(url, dst):
                    count += 1
        return count
    except Exception as e:
        return 0

In [ ]:
# ============================================================
# SCRAPER 2: Wikimedia Commons
# ============================================================
def scrape_wikimedia(search_term, dest_dir, max_images=10):
    try:
        params = {
            "action": "query",
            "list": "search",
            "srsearch": f"{search_term} filetype:png OR filetype:jpg",
            "srnamespace": "6",
            "format": "json",
            "srlimit": max_images * 2
        }
        r = requests.get("https://commons.wikimedia.org/w/api.php", params=params, timeout=15)
        data = r.json()
        existing = set(os.listdir(dest_dir)) if os.path.exists(dest_dir) else set()
        count = 0
        for result in data.get("query", {}).get("search", []):
            if count >= max_images:
                break
            title = result.get("title", "")
            if not title.startswith("File:"):
                continue
            img_params = {
                "action": "query",
                "titles": title,
                "prop": "imageinfo",
                "iiprop": "url",
                "format": "json",
            }
            ir = requests.get("https://commons.wikimedia.org/w/api.php", params=img_params, timeout=10)
            idata = ir.json()
            pages = idata.get("query", {}).get("pages", {})
            for pid, pinfo in pages.items():
                if pid == "-1":
                    continue
                imageinfo = pinfo.get("imageinfo", [])
                if imageinfo:
                    url = imageinfo[0].get("url", "")
                    if url:
                        fname = safe_filename(url, len(existing) + count + 1)
                        dst = dest_dir / fname
                        if not dst.exists() and download_image(url, dst):
                            count += 1
        return count
    except Exception as e:
        return 0

In [ ]:
# ============================================================
# MAIN SCRAPER: runs only for LEVELS_TO_SCRAPE
# ============================================================

LEVEL_NAMES = {int(k): v["name"] for k, v in SEARCH_QUERIES.items()}
SUBLEVEL_LABELS = ["Mastery", "Easier_Remediation", "Further_Remediation"]

os.makedirs(BASE, exist_ok=True)
grand_total = 0

print("=" * 60)
print(f"  FLN SPECIFIC LEVEL SCRAPER")
print(f"  Levels to scrape: {LEVELS_TO_SCRAPE}")
print(f"  Target: {IMAGES_PER_SUBLEVEL} images per sub-level")
print(f"  Output: {BASE}/pinterest_images/")
print("=" * 60)

for level_num in LEVELS_TO_SCRAPE:
    level_str = str(level_num)
    info = SEARCH_QUERIES.get(level_str)
    
    if level_num in REVIEW_ASSESSMENT_LEVELS:
        src_start, src_end = REVIEW_ASSESSMENT_LEVELS[level_num]
        name = f"Review Assessment {level_num}"
    elif info:
        name = info["name"]
        src_start = src_end = None
    else:
        print(f"\nLevel {level_num}: no search queries defined, skipping")
        continue
    
    safe_name = name.replace(" ", "_").replace("+", "_").replace(",", "").replace("-", "_")
    level_dir = BASE / "pinterest_images" / f"Level_{level_num:02d}_{safe_name}"
    
    print(f"\nLevel {level_num}: {name}")
    print("-" * 50)
    level_total = 0
    
    for sub_num in range(3):
        sub_label = SUBLEVEL_LABELS[sub_num]
        sub_dir = level_dir / f"{level_num}.{sub_num}_{sub_label}"
        os.makedirs(sub_dir, exist_ok=True)
        
        if level_num in REVIEW_ASSESSMENT_LEVELS:
            # Copy from source levels instead of scraping
            pinterest_dir = BASE / "pinterest_images"
            copied = 0
            for src_lvl in range(src_start, src_end + 1):
                # Find the source level directory by globbing
                src_dirs = list(pinterest_dir.glob(f"Level_{src_lvl:02d}_*"))
                if not src_dirs:
                    continue
                src_level_dir = src_dirs[0]
                # Find matching sub-level dir inside
                src_sub_dirs = list(src_level_dir.glob(f"{src_lvl}.{sub_num}_*"))
                if not src_sub_dirs:
                    continue
                src_sub_dir = src_sub_dirs[0]
                src_images = [f for f in src_sub_dir.iterdir() if f.suffix.lower() in VALID_EXTENSIONS]
                if not src_images:
                    continue
                # Copy each image into the review sub-dir
                for src_img in src_images:
                    dst_name = f"L{src_lvl:02d}_{src_img.name}"
                    dst_path = sub_dir / dst_name
                    if not dst_path.exists():
                        shutil.copy2(src_img, dst_path)
                        copied += 1
            final_count = len([f for f in sub_dir.iterdir() if f.suffix.lower() in VALID_EXTENSIONS])
            print(f"    -> {level_num}.{sub_num}: copied {copied} new, {final_count} total")
        else:
            # Standard query-based: fill sub-level with queries
            queries = info["queries"]
            existing_count = len([f for f in os.listdir(sub_dir) if f.endswith(tuple(VALID_EXTENSIONS))])
            needed = max(0, IMAGES_PER_SUBLEVEL - existing_count)
            if needed <= 0:
                print(f"  {level_num}.{sub_num} already has {existing_count} images, skipping")
                level_total += existing_count
                continue
            print(f"  {level_num}.{sub_num} [{sub_label}] needs {needed} more...")
            sub_queries = [f"{q} {'practice' if sub_num == 0 else 'easy' if sub_num == 1 else 'beginner'}" for q in queries]
            downloaded = 0
            attempts = 0
            while downloaded < needed and attempts < len(sub_queries) * 2:
                q = sub_queries[attempts % len(sub_queries)]
                attempts += 1
                remaining = needed - downloaded
                if remaining > 0:
                    n = scrape_duckduckgo(q, sub_dir, max_images=min(remaining + 2, 6))
                    downloaded += n
                    if n > 0:
                        print(f"    DDG -> {n} images")
                remaining = needed - downloaded
                if remaining > 0:
                    n = scrape_wikimedia(q, sub_dir, max_images=min(remaining + 2, 4))
                    downloaded += n
                    if n > 0:
                        print(f"    Wiki -> {n} images")
                time.sleep(0.5)
            # Rename sequentially
            files = sorted([f for f in os.listdir(sub_dir) if f.endswith(tuple(VALID_EXTENSIONS))])
            for i, fname in enumerate(files, 1):
                ext = os.path.splitext(fname)[1]
                new_name = f"{i:03d}{ext}"
                if fname != new_name:
                    os.rename(sub_dir / fname, sub_dir / new_name)
            final_count = len([f for f in os.listdir(sub_dir) if f.endswith(tuple(VALID_EXTENSIONS))])
            print(f"    -> {level_num}.{sub_num}: {final_count} total images")
        level_total += final_count
    
    print(f"  Level total: {level_total} images")
    grand_total += level_total

print(f"\n{'=' * 60}")
print(f"  GRAND TOTAL: {grand_total} images downloaded")
print(f"{'=' * 60}")
print(f"\nOutput: {BASE}/pinterest_images/")

In [ ]:
# ============================================================
# GENERATE Q No. | ILLUSTRATION | QUESTION TABLE (HTML, Cambria, Docs-ready)
# ============================================================
from collections import OrderedDict, Counter

def get_subdirs(dir_path):
    subdirs = sorted([d for d in dir_path.iterdir() if d.is_dir()])
    result = []
    for d in subdirs:
        if d.name.startswith("_"):
            result.extend(sorted([s for s in d.iterdir() if s.is_dir()]))
        else:
            result.append(d)
    return result

def generate_table():
    rows = []
    pinterest_dir = BASE / "pinterest_images"
    if not pinterest_dir.exists():
        print("No pinterest_images directory found.")
        return []
    for level_dir in sorted(pinterest_dir.iterdir()):
        if not level_dir.is_dir():
            continue
        for sub_dir in get_subdirs(level_dir):
            parts = sub_dir.name.split("_", 1)
            sub_code = parts[0] if parts else ""
            images = sorted([f for f in sub_dir.iterdir()
                           if f.suffix.lower() in VALID_EXTENSIONS])
            classify = "mastery" if ".0" in sub_code else (
                "remediation_easier" if ".1" in sub_code else "remediation_further"
            )
            level_num = sub_code.split(".")[0] if "." in sub_code else ""
            for i, img_path in enumerate(images, 1):
                rows.append({
                    "q_no": i,
                    "illustration": str(img_path.relative_to(BASE)),
                    "level": level_num,
                    "sub_level": sub_code,
                    "classification": classify,
                })
    return rows

def export_html(rows):
    if not rows:
        print("No rows to export")
        return
    levels = OrderedDict()
    for r in rows:
        key = (r["level"], r["sub_level"], r["classification"])
        levels.setdefault(key, []).append(r)
    html_parts = ['<html><head><meta charset="utf-8">']
    html_parts.append('<style>')
    html_parts.append('body { font-family: "Cambria", serif; font-size: 11pt; margin: 20px; }')
    html_parts.append('h2 { font-family: "Cambria", serif; color: #1a5276; }')
    html_parts.append('table { border-collapse: collapse; width: 100%; margin-bottom: 30px; }')
    html_parts.append('th, td { border: 1px solid #999; padding: 6px 10px; vertical-align: middle; font-family: "Cambria", serif; }')
    html_parts.append('th { background-color: #d4e6f1; font-weight: bold; }')
    html_parts.append('img { max-width: 120px; max-height: 120px; }')
    html_parts.append('</style></head><body>')
    for key, group in levels.items():
        level_num, sub_code, classify = key
        label = classify.replace("_", " ").title()
        html_parts.append(f'<h2>Level {level_num} — Sub-level {sub_code} ({label})</h2>')
        html_parts.append('<table>')
        html_parts.append('<tr><th>Q No.</th><th>Illustration</th><th>Question</th></tr>')
        for r in group:
            img_abs = str(BASE / r["illustration"])
            html_parts.append(
                f'<tr><td>{r["q_no"]}</td>'
                f'<td><img src="file://{img_abs}"></td>'
                f'<td></td></tr>'
            )
        html_parts.append('</table>')
    html_parts.append('</body></html>')
    out_path = BASE / "fln_tables.html"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(html_parts))
    print(f"Exported {len(rows)} rows to {out_path}")
    print("Open in browser, Ctrl+A -> Ctrl+C -> paste into Google Docs")

print("\nGenerating Q No. | Illustration | Question table...")
rows = generate_table()
if rows:
    export_html(rows)
    sub_counts = Counter()
    level_counts = Counter()
    for r in rows:
        sub_counts[r["sub_level"]] += 1
        level_counts[r["level"]] += 1
    print(f"\n{'='*60}")
    print(f"  TABLE SUMMARY")
    print(f"{'='*60}")
    print(f"  Total rows: {len(rows)}")
    print(f"  Levels: {len(level_counts)}, Sub-levels: {len(sub_counts)}")
    low = [(k, v) for k, v in sorted(sub_counts.items()) if v < 10]
    if low:
        print(f"  Sub-levels with < 10 images ({len(low)}):")
        for k, v in low:
            print(f"    {k}: {v}")
    else:
        print(f"  All sub-levels have >= 10 images!")
    print(f"\nDone: {BASE / 'fln_tables.html'}")

### To Use:
1. Run all cells
2. Answer the prompts in the terminal for levels/review config
3. Images are scraped or copied automatically
4. Table is generated at the end as `fln_tables.html`
5. Download from Colab's Files tab or run:
```python
from google.colab import files
import zipfile, os
!zip -r /content/fln_output.zip /content/fln_output/pinterest_images/
files.download("/content/fln_output.zip")
```